In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Ruta del archivo original
input_path = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\train_full_inicial.csv")

# Cargar dataset
df = pd.read_csv(input_path)

# Recalcular presión de pulso desde PAS - PAD
df["pp_recalc"] = df["systolic_bp"] - df["diastolic_bp"]

# Flag: PA faltante
df["bp_missing_original"] = df["systolic_bp"].isna() | df["diastolic_bp"].isna()

# Categorizar presión de pulso recalculada
conditions = [
    df["bp_missing_original"],
    df["pp_recalc"] <= 0,
    (df["pp_recalc"] > 0) & (df["pp_recalc"] < 10),
    (df["pp_recalc"] >= 10) & (df["pp_recalc"] < 20),
    (df["pp_recalc"] >= 20) & (df["pp_recalc"] < 30),
    (df["pp_recalc"] >= 30) & (df["pp_recalc"] <= 60),
    (df["pp_recalc"] > 60) & (df["pp_recalc"] <= 100),
    df["pp_recalc"] > 100
]

choices = [
    "bp_missing",
    "pp_impossible",
    "pp_extremely_low",
    "pp_very_low",
    "pp_low_plausible",
    "pp_usual",
    "pp_wide",
    "pp_extremely_wide"
]

df["pp_category"] = np.select(
    conditions,
    choices,
    default="pp_unclassified"
)

# Crear flags binarios
df["pp_impossible_flag"] = df["pp_category"].eq("pp_impossible").astype(int)
df["pp_extremely_low_flag"] = df["pp_category"].eq("pp_extremely_low").astype(int)
df["pp_very_low_flag"] = df["pp_category"].eq("pp_very_low").astype(int)
df["pp_low_flag"] = df["pp_category"].isin(["pp_extremely_low", "pp_very_low", "pp_low_plausible"]).astype(int)
df["pp_wide_flag"] = df["pp_category"].eq("pp_wide").astype(int)
df["pp_extremely_wide_flag"] = df["pp_category"].eq("pp_extremely_wide").astype(int)

# Resumen inicial
resumen_pp = (
    df["pp_category"]
    .value_counts(dropna=False)
    .rename_axis("pp_category")
    .reset_index(name="n")
)

resumen_pp["%"] = (resumen_pp["n"] / len(df) * 100).round(2)

resumen_pp

,pp_category,n,%
0,pp_usual,35517,44.40
1,pp_wide,21373,26.72
2,pp_low_plausible,7996,9.99
3,pp_very_low,5038,6.30
4,bp_missing,4146,5.18
5,pp_extremely_low,2703,3.38
6,pp_impossible,2105,2.63
7,pp_extremely_wide,1122,1.40


In [2]:
conditions = [
    df["bp_missing_original"],
    df["pp_recalc"] <= 0,
    (df["pp_recalc"] > 0) & (df["pp_recalc"] < 10),
    (df["pp_recalc"] >= 10) & (df["pp_recalc"] < 20),
    (df["pp_recalc"] >= 20) & (df["pp_recalc"] < 30),
    (df["pp_recalc"] >= 30) & (df["pp_recalc"] <= 60),
    (df["pp_recalc"] > 60) & (df["pp_recalc"] <= 100),
    df["pp_recalc"] > 100
]

choices = [
    "00_bp_missing",
    "01_pp_le_0_impossible",
    "02_pp_0_10_extremely_low",
    "03_pp_10_20_very_low",
    "04_pp_20_30_low_plausible",
    "05_pp_30_60_usual",
    "06_pp_60_100_wide",
    "07_pp_gt_100_extremely_wide"
]

df["pp_category"] = np.select(
    conditions,
    choices,
    default="99_pp_unclassified"
)

In [3]:
df["pp_impossible_flag"] = df["pp_category"].eq("01_pp_le_0_impossible").astype(int)

df["pp_extremely_low_flag"] = df["pp_category"].eq("02_pp_0_10_extremely_low").astype(int)

df["pp_very_low_flag"] = df["pp_category"].eq("03_pp_10_20_very_low").astype(int)

df["pp_low_flag"] = df["pp_category"].isin([
    "02_pp_0_10_extremely_low",
    "03_pp_10_20_very_low",
    "04_pp_20_30_low_plausible"
]).astype(int)

df["pp_wide_flag"] = df["pp_category"].eq("06_pp_60_100_wide").astype(int)

df["pp_extremely_wide_flag"] = df["pp_category"].eq("07_pp_gt_100_extremely_wide").astype(int)

In [4]:
pp_category_labels = {
    "00_bp_missing": "PA faltante",
    "01_pp_le_0_impossible": "PP ≤ 0 imposible",
    "02_pp_0_10_extremely_low": "PP 0-10 extremadamente baja",
    "03_pp_10_20_very_low": "PP 10-20 muy baja",
    "04_pp_20_30_low_plausible": "PP 20-30 baja plausible",
    "05_pp_30_60_usual": "PP 30-60 usual",
    "06_pp_60_100_wide": "PP 60-100 amplia",
    "07_pp_gt_100_extremely_wide": "PP >100 extremadamente amplia",
    "99_pp_unclassified": "No clasificada"
}

df["pp_category_label"] = df["pp_category"].map(pp_category_labels)

In [5]:
resumen_pp = (
    df["pp_category_label"]
    .value_counts(dropna=False)
    .rename_axis("pp_category_label")
    .reset_index(name="n")
)

resumen_pp["%"] = (resumen_pp["n"] / len(df) * 100).round(2)

resumen_pp

,pp_category_label,n,%
0,PP 30-60 usual,35517,44.40
1,PP 60-100 amplia,21373,26.72
2,PP 20-30 baja plausible,7996,9.99
3,PP 10-20 muy baja,5038,6.30
4,PA faltante,4146,5.18
5,PP 0-10 extremadamente baja,2703,3.38
6,PP ≤ 0 imposible,2105,2.63
7,PP >100 extremadamente amplia,1122,1.40


In [6]:
columnas_revision = [
    "systolic_bp",
    "diastolic_bp",
    "pp_recalc",
    "pp_category_label",
    "heart_rate",
    "respiratory_rate",
    "spo2",
    "gcs_total",
    "mental_status_triage",
    "shock_index",
    "news2_score",
    "triage_acuity",
    "disposition"
]

df.loc[
    df["pp_category"].eq("01_pp_le_0_impossible"),
    columnas_revision
].head(20)

,systolic_bp,diastolic_bp,pp_recalc,pp_category_label,heart_rate,respiratory_rate,spo2,gcs_total,mental_status_triage,shock_index,news2_score,triage_acuity,disposition
30,112.3,122.2,-9.9,PP ≤ 0 imposible,79.6,17.3,90.9,15,alert,0.709,3,3,transferred
59,111.3,113.7,-2.4,PP ≤ 0 imposible,99.6,20.8,97.1,15,confused,0.895,4,3,discharged
82,76.1,83.1,-7.0,PP ≤ 0 imposible,89.9,28.0,89.5,14,unresponsive,1.181,14,2,admitted
155,107.4,108.0,-0.6,PP ≤ 0 imposible,92.6,23.7,97.9,15,confused,0.862,4,3,lwbs
190,78.2,82.3,-4.1,PP ≤ 0 imposible,95.9,18.8,93.8,15,confused,1.226,5,3,discharged
196,97.2,115.9,-18.7,PP ≤ 0 imposible,95.0,17.9,92.4,15,drowsy,0.977,5,3,admitted
256,58.5,74.3,-15.8,PP ≤ 0 imposible,82.7,26.5,90.7,9,unresponsive,1.414,14,2,lwbs
278,97.1,118.5,-21.4,PP ≤ 0 imposible,99.0,16.1,99.9,15,confused,1.020,3,3,discharged
381,58.6,68.1,-9.5,PP ≤ 0 imposible,79.3,NaN,97.8,15,alert,1.353,3,5,discharged
390,67.2,67.8,-0.6,PP ≤ 0 imposible,95.2,22.4,90.8,10,confused,1.417,14,2,admitted


In [7]:
df.loc[
    df["pp_category"].eq("02_pp_0_10_extremely_low"),
    columnas_revision
].head(7)

,systolic_bp,diastolic_bp,pp_recalc,pp_category_label,heart_rate,respiratory_rate,spo2,gcs_total,mental_status_triage,shock_index,news2_score,triage_acuity,disposition
9,84.9,78.3,6.6,PP 0-10 extremadamente baja,86.4,13.8,98.5,15,alert,1.018,3,4,discharged
41,94.5,89.4,5.1,PP 0-10 extremadamente baja,89.4,21.0,100.0,15,alert,0.946,4,3,admitted
49,81.7,72.0,9.7,PP 0-10 extremadamente baja,79.2,17.9,89.5,15,alert,0.969,6,3,transferred
52,72.0,70.1,1.9,PP 0-10 extremadamente baja,93.4,19.9,97.3,15,agitated,1.297,4,2,discharged
77,97.9,92.3,5.6,PP 0-10 extremadamente baja,76.8,16.1,97.2,15,confused,0.784,2,4,discharged
84,97.7,92.9,4.8,PP 0-10 extremadamente baja,96.1,19.9,97.3,15,confused,0.984,5,3,discharged
98,104.3,96.8,7.5,PP 0-10 extremadamente baja,93.9,10.0,99.1,15,alert,0.900,3,3,lwbs


In [8]:
df.loc[
    df["pp_category"].eq("03_pp_10_20_very_low"),
    columnas_revision
].head(10)

,systolic_bp,diastolic_bp,pp_recalc,pp_category_label,heart_rate,respiratory_rate,spo2,gcs_total,mental_status_triage,shock_index,news2_score,triage_acuity,disposition
2,94.7,83.3,11.4,PP 10-20 muy baja,75.6,14.7,100.0,15,alert,0.798,2,5,discharged
34,78.5,62.6,15.9,PP 10-20 muy baja,123.0,21.9,95.5,10,drowsy,1.567,10,2,discharged
46,98.6,82.8,15.8,PP 10-20 muy baja,109.3,15.5,98.2,15,drowsy,1.109,5,3,transferred
61,125.7,109.1,16.6,PP 10-20 muy baja,110.4,14.8,97.0,15,alert,0.878,2,4,admitted
78,61.9,44.9,17.0,PP 10-20 muy baja,104.7,29.3,99.4,4,drowsy,1.691,11,1,admitted
89,76.6,57.2,19.4,PP 10-20 muy baja,94.9,29.5,86.5,13,agitated,1.239,14,2,admitted
100,97.7,78.9,18.8,PP 10-20 muy baja,109.1,17.4,97.1,15,alert,1.117,3,3,lwbs
104,79.0,63.5,15.5,PP 10-20 muy baja,125.7,30.8,100.0,11,confused,1.591,12,2,admitted
117,70.2,59.3,10.9,PP 10-20 muy baja,139.9,18.5,94.8,11,drowsy,1.993,12,2,admitted
124,97.2,83.0,14.2,PP 10-20 muy baja,84.1,15.1,100.0,15,alert,0.865,2,4,discharged


In [9]:
tabla_pp_esi = pd.crosstab(
    df["pp_category_label"],
    df["triage_acuity"],
    normalize="index"
).round(3) * 100

tabla_pp_esi

triage_acuity,1,2,3,4,5
pp_category_label,,,,,
PA faltante,0.0,0.0,0.0,67.4,32.6
PP 0-10 extremadamente baja,11.1,33.1,38.0,11.6,6.2
PP 10-20 muy baja,9.1,28.3,37.2,16.6,8.8
PP 20-30 baja plausible,7.3,24.5,35.8,21.2,11.2
PP 30-60 usual,3.8,16.9,35.3,28.2,15.8
PP 60-100 amplia,1.3,10.4,42.5,32.5,13.3
PP >100 extremadamente amplia,0.1,6.1,65.8,23.7,4.3
PP ≤ 0 imposible,11.9,40.0,37.8,7.1,3.2


In [10]:
tabla_pp_esi_n = pd.crosstab(
    df["pp_category_label"],
    df["triage_acuity"]
)

tabla_pp_esi_n

triage_acuity,1,2,3,4,5
pp_category_label,,,,,
PA faltante,0,0,0,2796,1350
PP 0-10 extremadamente baja,300,895,1028,313,167
PP 10-20 muy baja,456,1428,1876,837,441
PP 20-30 baja plausible,587,1957,2861,1696,895
PP 30-60 usual,1346,6016,12534,10024,5597
PP 60-100 amplia,281,2232,9088,6939,2833
PP >100 extremadamente amplia,1,69,738,266,48
PP ≤ 0 imposible,251,842,796,149,67


In [11]:
tabla_pp_disposition = pd.crosstab(
    df["pp_category_label"],
    df["disposition"],
    normalize="index"
).round(3) * 100

tabla_pp_disposition

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
pp_category_label,,,,,,,
PA faltante,7.0,0.0,76.3,4.6,7.0,2.3,2.8
PP 0-10 extremadamente baja,44.2,1.2,33.3,2.3,3.7,6.0,9.2
PP 10-20 muy baja,40.8,1.0,37.4,2.6,3.6,5.9,8.6
PP 20-30 baja plausible,37.2,0.8,42.1,3.0,3.7,5.9,7.3
PP 30-60 usual,30.4,0.5,49.2,3.6,4.5,5.4,6.4
PP 60-100 amplia,27.6,0.2,52.0,3.7,5.1,5.5,5.9
PP >100 extremadamente amplia,30.7,0.0,46.3,4.6,4.6,6.3,7.4
PP ≤ 0 imposible,49.4,1.8,27.6,2.0,2.4,7.2,9.6


In [12]:
tabla_pp_disposition_n = pd.crosstab(
    df["pp_category_label"],
    df["disposition"]
)

tabla_pp_disposition_n

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
pp_category_label,,,,,,,
PA faltante,290,0,3164,190,289,96,117
PP 0-10 extremadamente baja,1196,33,899,63,100,163,249
PP 10-20 muy baja,2058,52,1886,129,183,298,432
PP 20-30 baja plausible,2972,66,3370,239,292,470,587
PP 30-60 usual,10802,185,17487,1262,1592,1919,2270
PP 60-100 amplia,5898,37,11123,786,1097,1169,1263
PP >100 extremadamente amplia,345,0,519,52,52,71,83
PP ≤ 0 imposible,1040,38,580,43,51,151,202


In [13]:
resumen_clinico_pp = (
    df.groupby("pp_category_label")[[
        "systolic_bp",
        "diastolic_bp",
        "pp_recalc",
        "heart_rate",
        "respiratory_rate",
        "spo2",
        "gcs_total",
        "shock_index",
        "news2_score"
    ]]
    .median()
    .round(2)
)

resumen_clinico_pp

,systolic_bp,diastolic_bp,pp_recalc,heart_rate,respiratory_rate,spo2,gcs_total,shock_index,news2_score
pp_category_label,,,,,,,,,
PA faltante,NaN,NaN,NaN,81.5,15.6,98.4,15.0,NaN,0.0
PP 0-10 extremadamente baja,87.3,81.9,5.9,96.8,19.2,95.1,15.0,1.11,6.0
PP 10-20 muy baja,96.4,81.0,15.6,94.2,18.5,95.9,15.0,0.97,4.0
PP 20-30 baja plausible,105.1,79.8,25.4,92.3,18.1,96.3,15.0,0.87,3.0
PP 30-60 usual,121.9,76.6,45.4,89.2,17.3,97.1,15.0,0.73,1.0
PP 60-100 amplia,143.7,70.8,71.4,88.7,17.0,97.2,15.0,0.61,1.0
PP >100 extremadamente amplia,170.7,61.4,107.2,90.8,17.5,96.5,15.0,0.53,1.0
PP ≤ 0 imposible,73.9,83.6,-7.1,99.6,20.0,94.4,15.0,1.36,8.0


In [14]:
df_missing_bp = df[df["bp_missing_original"] == True]

df_missing_bp[[
    "triage_acuity",
    "mental_status_triage",
    "disposition",
    "heart_rate",
    "respiratory_rate",
    "spo2",
    "gcs_total",
    "news2_score"
]].describe(include="all")

,triage_acuity,mental_status_triage,disposition,heart_rate,respiratory_rate,spo2,gcs_total,news2_score
count,4146.000000,4146,4146,4146.000000,3786.000000,4146.000000,4146.0,4146.000000
unique,NaN,4,6,NaN,NaN,NaN,NaN,NaN
top,NaN,alert,discharged,NaN,NaN,NaN,NaN,NaN
freq,NaN,3453,3164,NaN,NaN,NaN,NaN,NaN
mean,4.325615,NaN,NaN,81.746358,15.665320,98.145562,15.0,0.421129
std,0.468661,NaN,NaN,12.910359,2.064666,1.566227,0.0,0.657280
min,4.000000,NaN,NaN,37.600000,9.600000,91.900000,15.0,0.000000
25%,4.000000,NaN,NaN,73.125000,14.200000,97.200000,15.0,0.000000
50%,4.000000,NaN,NaN,81.500000,15.600000,98.400000,15.0,0.000000
75%,5.000000,NaN,NaN,90.400000,17.100000,99.500000,15.0,1.000000


In [15]:
pd.crosstab(
    df["bp_missing_original"],
    df["triage_acuity"],
    normalize="index"
).round(3) * 100

triage_acuity,1,2,3,4,5
bp_missing_original,,,,,
False,4.2,17.7,38.1,26.7,13.2
True,0.0,0.0,0.0,67.4,32.6


In [16]:
pd.crosstab(
    df["bp_missing_original"],
    df["disposition"],
    normalize="index"
).round(3) * 100

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
bp_missing_original,,,,,,,
False,32.0,0.5,47.3,3.4,4.4,5.6,6.7
True,7.0,0.0,76.3,4.6,7.0,2.3,2.8


In [17]:
hx_cols = [col for col in df.columns if col.startswith("hx_")]

hx_cols

['hx_hypertension',
 'hx_diabetes_type2',
 'hx_diabetes_type1',
 'hx_asthma',
 'hx_copd',
 'hx_heart_failure',
 'hx_atrial_fibrillation',
 'hx_ckd',
 'hx_liver_disease',
 'hx_malignancy',
 'hx_obesity',
 'hx_depression',
 'hx_anxiety',
 'hx_dementia',
 'hx_epilepsy',
 'hx_hypothyroidism',
 'hx_hyperthyroidism',
 'hx_hiv',
 'hx_coagulopathy',
 'hx_immunosuppressed',
 'hx_pregnant',
 'hx_substance_use_disorder',
 'hx_coronary_artery_disease',
 'hx_stroke_prior',
 'hx_peripheral_vascular_disease']

In [18]:
top_hx_por_pp = []

for categoria in df["pp_category_label"].dropna().unique():
    temp = df[df["pp_category_label"] == categoria]
    
    resumen = (
        temp[hx_cols]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    
    resumen.columns = ["antecedente", "prevalencia_en_categoria"]
    resumen["pp_category_label"] = categoria
    resumen["n_categoria"] = len(temp)
    resumen["prevalencia_en_categoria_%"] = (resumen["prevalencia_en_categoria"] * 100).round(2)
    
    top_hx_por_pp.append(resumen)

top_hx_por_pp = pd.concat(top_hx_por_pp, ignore_index=True)

top_hx_por_pp = top_hx_por_pp[
    ["pp_category_label", "n_categoria", "antecedente", "prevalencia_en_categoria_%"]
]

top_hx_por_pp.head(20)

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%
0,PP 20-30 baja plausible,7996,hx_atrial_fibrillation,27.15
1,PP 20-30 baja plausible,7996,hx_hypertension,27.14
2,PP 20-30 baja plausible,7996,hx_diabetes_type2,26.79
3,PP 20-30 baja plausible,7996,hx_ckd,26.69
4,PP 20-30 baja plausible,7996,hx_coronary_artery_disease,26.15
5,PP 20-30 baja plausible,7996,hx_heart_failure,26.04
6,PP 20-30 baja plausible,7996,hx_stroke_prior,25.21
7,PP 20-30 baja plausible,7996,hx_immunosuppressed,20.32
8,PP 20-30 baja plausible,7996,hx_depression,20.30
9,PP 20-30 baja plausible,7996,hx_copd,20.22


In [19]:
# Prevalencia global de cada antecedente
prevalencia_global = df[hx_cols].mean()

enriquecimiento_hx_por_pp = []

for categoria in df["pp_category_label"].dropna().unique():
    temp = df[df["pp_category_label"] == categoria]
    
    prevalencia_categoria = temp[hx_cols].mean()
    
    resumen = pd.DataFrame({
        "antecedente": hx_cols,
        "prevalencia_global": prevalencia_global.values,
        "prevalencia_en_categoria": prevalencia_categoria.values
    })
    
    resumen["diferencia_absoluta"] = resumen["prevalencia_en_categoria"] - resumen["prevalencia_global"]
    
    resumen["ratio_vs_global"] = (
        resumen["prevalencia_en_categoria"] / resumen["prevalencia_global"]
    )
    
    resumen["pp_category_label"] = categoria
    resumen["n_categoria"] = len(temp)
    
    resumen["prevalencia_global_%"] = (resumen["prevalencia_global"] * 100).round(2)
    resumen["prevalencia_en_categoria_%"] = (resumen["prevalencia_en_categoria"] * 100).round(2)
    resumen["diferencia_absoluta_%"] = (resumen["diferencia_absoluta"] * 100).round(2)
    resumen["ratio_vs_global"] = resumen["ratio_vs_global"].round(2)
    
    resumen = resumen.sort_values(
        ["diferencia_absoluta", "ratio_vs_global"],
        ascending=False
    ).head(10)
    
    enriquecimiento_hx_por_pp.append(resumen)

enriquecimiento_hx_por_pp = pd.concat(enriquecimiento_hx_por_pp, ignore_index=True)

enriquecimiento_hx_por_pp = enriquecimiento_hx_por_pp[
    [
        "pp_category_label",
        "n_categoria",
        "antecedente",
        "prevalencia_en_categoria_%",
        "prevalencia_global_%",
        "diferencia_absoluta_%",
        "ratio_vs_global"
    ]
]


enriquecimiento_hx_por_pp



,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global
0,PP 20-30 baja plausible,7996,hx_depression,20.30,19.54,0.76,1.04
1,PP 20-30 baja plausible,7996,hx_hiv,20.12,19.56,0.56,1.03
2,PP 20-30 baja plausible,7996,hx_copd,20.22,19.66,0.56,1.03
3,PP 20-30 baja plausible,7996,hx_pregnant,4.06,3.54,0.53,1.15
4,PP 20-30 baja plausible,7996,hx_asthma,20.09,19.65,0.44,1.02
...,...,...,...,...,...,...,...
75,PP ≤ 0 imposible,2105,hx_anxiety,20.48,19.87,0.61,1.03
76,PP ≤ 0 imposible,2105,hx_hiv,20.05,19.56,0.49,1.02
77,PP ≤ 0 imposible,2105,hx_malignancy,20.29,19.86,0.42,1.02
78,PP ≤ 0 imposible,2105,hx_substance_use_disorder,20.14,19.88,0.26,1.01


In [20]:
categorias_problematicas = [
    'PP 20-30 baja plausible', 'PP 30-60 usual', 'PP 10-20 muy baja',
       'PP 60-100 amplia', 'PA faltante', 'PP 0-10 extremadamente baja',
       'PP >100 extremadamente amplia', 'PP ≤ 0 imposible'
]

enriquecimiento_problematicas = enriquecimiento_hx_por_pp[
    enriquecimiento_hx_por_pp["pp_category_label"].isin(categorias_problematicas)
]

enriquecimiento_problematicas

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global
0,PP 20-30 baja plausible,7996,hx_depression,20.30,19.54,0.76,1.04
1,PP 20-30 baja plausible,7996,hx_hiv,20.12,19.56,0.56,1.03
2,PP 20-30 baja plausible,7996,hx_copd,20.22,19.66,0.56,1.03
3,PP 20-30 baja plausible,7996,hx_pregnant,4.06,3.54,0.53,1.15
4,PP 20-30 baja plausible,7996,hx_asthma,20.09,19.65,0.44,1.02
...,...,...,...,...,...,...,...
75,PP ≤ 0 imposible,2105,hx_anxiety,20.48,19.87,0.61,1.03
76,PP ≤ 0 imposible,2105,hx_hiv,20.05,19.56,0.49,1.02
77,PP ≤ 0 imposible,2105,hx_malignancy,20.29,19.86,0.42,1.02
78,PP ≤ 0 imposible,2105,hx_substance_use_disorder,20.14,19.88,0.26,1.01


In [21]:
df["pp_category_label"].unique()

array(['PP 20-30 baja plausible', 'PP 30-60 usual', 'PP 10-20 muy baja',
       'PP 60-100 amplia', 'PA faltante', 'PP 0-10 extremadamente baja',
       'PP >100 extremadamente amplia', 'PP ≤ 0 imposible'], dtype=object)

In [22]:
tabla_fa_pp = (
    df.groupby("pp_category_label")["hx_atrial_fibrillation"]
    .agg(
        n_categoria="count",
        n_con_fa="sum",
        prevalencia_fa="mean"
    )
    .reset_index()
)

tabla_fa_pp["prevalencia_fa_%"] = (tabla_fa_pp["prevalencia_fa"] * 100).round(2)

tabla_fa_pp = tabla_fa_pp[
    ["pp_category_label", "n_categoria", "n_con_fa", "prevalencia_fa_%"]
]

tabla_fa_pp

,pp_category_label,n_categoria,n_con_fa,prevalencia_fa_%
0,PA faltante,4146,1109,26.75
1,PP 0-10 extremadamente baja,2703,715,26.45
2,PP 10-20 muy baja,5038,1372,27.23
3,PP 20-30 baja plausible,7996,2171,27.15
4,PP 30-60 usual,35517,9907,27.89
5,PP 60-100 amplia,21373,6317,29.56
6,PP >100 extremadamente amplia,1122,417,37.17
7,PP ≤ 0 imposible,2105,562,26.70


In [23]:
tabla_fa_pp.sort_values("prevalencia_fa_%", ascending=False)

,pp_category_label,n_categoria,n_con_fa,prevalencia_fa_%
6,PP >100 extremadamente amplia,1122,417,37.17
5,PP 60-100 amplia,21373,6317,29.56
4,PP 30-60 usual,35517,9907,27.89
2,PP 10-20 muy baja,5038,1372,27.23
3,PP 20-30 baja plausible,7996,2171,27.15
0,PA faltante,4146,1109,26.75
7,PP ≤ 0 imposible,2105,562,26.70
1,PP 0-10 extremadamente baja,2703,715,26.45


In [24]:
pd.crosstab(
    df["hx_atrial_fibrillation"],
    df["pp_category_label"],
    normalize="index"
).round(3) * 100

pp_category_label,PA faltante,PP 0-10 extremadamente baja,PP 10-20 muy baja,PP 20-30 baja plausible,PP 30-60 usual,PP 60-100 amplia,PP >100 extremadamente amplia,PP ≤ 0 imposible
hx_atrial_fibrillation,,,,,,,,
0,5.3,3.5,6.4,10.1,44.6,26.2,1.2,2.7
1,4.9,3.2,6.1,9.6,43.9,28.0,1.8,2.5


In [25]:
pd.crosstab(
    df["hx_atrial_fibrillation"],
    df["pp_category_label"]
)

pp_category_label,PA faltante,PP 0-10 extremadamente baja,PP 10-20 muy baja,PP 20-30 baja plausible,PP 30-60 usual,PP 60-100 amplia,PP >100 extremadamente amplia,PP ≤ 0 imposible
hx_atrial_fibrillation,,,,,,,,
0,3037,1988,3666,5825,25610,15056,705,1543
1,1109,715,1372,2171,9907,6317,417,562


In [26]:
def revisar_hx_vs_pp(df, hx_col):
    tabla = (
        df.groupby("pp_category_label")[hx_col]
        .agg(
            n_categoria="count",
            n_con_antecedente="sum",
            prevalencia="mean"
        )
        .reset_index()
    )
    
    tabla["prevalencia_%"] = (tabla["prevalencia"] * 100).round(2)
    
    tabla = tabla[
        ["pp_category_label", "n_categoria", "n_con_antecedente", "prevalencia_%"]
    ].sort_values("prevalencia_%", ascending=False)
    
    return tabla

In [27]:
revisar_hx_vs_pp(df, "hx_atrial_fibrillation")

,pp_category_label,n_categoria,n_con_antecedente,prevalencia_%
6,PP >100 extremadamente amplia,1122,417,37.17
5,PP 60-100 amplia,21373,6317,29.56
4,PP 30-60 usual,35517,9907,27.89
2,PP 10-20 muy baja,5038,1372,27.23
3,PP 20-30 baja plausible,7996,2171,27.15
0,PA faltante,4146,1109,26.75
7,PP ≤ 0 imposible,2105,562,26.70
1,PP 0-10 extremadamente baja,2703,715,26.45


In [28]:
revisar_hx_vs_pp(df, "hx_heart_failure")

,pp_category_label,n_categoria,n_con_antecedente,prevalencia_%
6,PP >100 extremadamente amplia,1122,390,34.76
5,PP 60-100 amplia,21373,6440,30.13
4,PP 30-60 usual,35517,9854,27.74
0,PA faltante,4146,1122,27.06
7,PP ≤ 0 imposible,2105,564,26.79
2,PP 10-20 muy baja,5038,1319,26.18
1,PP 0-10 extremadamente baja,2703,705,26.08
3,PP 20-30 baja plausible,7996,2082,26.04


In [29]:
revisar_hx_vs_pp(df, "hx_ckd")

,pp_category_label,n_categoria,n_con_antecedente,prevalencia_%
6,PP >100 extremadamente amplia,1122,375,33.42
5,PP 60-100 amplia,21373,6575,30.76
4,PP 30-60 usual,35517,9807,27.61
0,PA faltante,4146,1121,27.04
3,PP 20-30 baja plausible,7996,2134,26.69
1,PP 0-10 extremadamente baja,2703,718,26.56
2,PP 10-20 muy baja,5038,1319,26.18
7,PP ≤ 0 imposible,2105,496,23.56


In [30]:
revisar_hx_vs_pp(df, "hx_ckd")

,pp_category_label,n_categoria,n_con_antecedente,prevalencia_%
6,PP >100 extremadamente amplia,1122,375,33.42
5,PP 60-100 amplia,21373,6575,30.76
4,PP 30-60 usual,35517,9807,27.61
0,PA faltante,4146,1121,27.04
3,PP 20-30 baja plausible,7996,2134,26.69
1,PP 0-10 extremadamente baja,2703,718,26.56
2,PP 10-20 muy baja,5038,1319,26.18
7,PP ≤ 0 imposible,2105,496,23.56


In [31]:
revisar_hx_vs_pp(df, "hx_coronary_artery_disease")

,pp_category_label,n_categoria,n_con_antecedente,prevalencia_%
6,PP >100 extremadamente amplia,1122,393,35.03
5,PP 60-100 amplia,21373,6510,30.46
4,PP 30-60 usual,35517,9930,27.96
0,PA faltante,4146,1116,26.92
2,PP 10-20 muy baja,5038,1332,26.44
3,PP 20-30 baja plausible,7996,2091,26.15
7,PP ≤ 0 imposible,2105,547,25.99
1,PP 0-10 extremadamente baja,2703,692,25.60


In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

# Columnas que NO queremos usar para este análisis
cols_excluir = [
    "patient_id",
    
    # PA y derivados directos
    "systolic_bp",
    "diastolic_bp",
    "mean_arterial_pressure",
    "pulse_pressure",
    "pp_recalc",
    "shock_index",
    "news2_score",
    
    # Outcomes posteriores
    "disposition",
    "ed_los_hours",
    
    # Referencia ESI: puedes excluirla aquí para no mezclar gravedad asignada
    "triage_acuity"
]

# Si tienes flags de PP, también excluirlas para no comparar PP contra PP
cols_excluir += [col for col in df.columns if col.startswith("pp_")]
cols_excluir += ["bp_missing_original", "pp_category", "pp_category_label"]

# Variables categóricas útiles
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Variables binarias tipo hx_*
hx_cols = [col for col in df.columns if col.startswith("hx_")]

# Otras variables binarias o categóricas numéricas que pueden servir
# Ejemplo: sex, age_group, arrival_mode, mental_status_triage, etc.
candidate_cols = []

for col in df.columns:
    if col in cols_excluir:
        continue
    
    # Excluir columnas con demasiados valores únicos, como texto libre
    if df[col].nunique(dropna=False) > 30:
        continue
    
    # Incluir categóricas o binarias/ordinales de baja cardinalidad
    candidate_cols.append(col)

candidate_cols

['site_id',
 'arrival_mode',
 'arrival_hour',
 'arrival_day',
 'arrival_month',
 'arrival_season',
 'shift',
 'age_group',
 'sex',
 'language',
 'insurance_type',
 'transport_origin',
 'pain_location',
 'mental_status_triage',
 'chief_complaint_system',
 'num_prior_ed_visits_12m',
 'num_prior_admissions_12m',
 'num_active_medications',
 'num_comorbidities',
 'gcs_total',
 'pain_score',
 'hx_hypertension',
 'hx_diabetes_type2',
 'hx_diabetes_type1',
 'hx_asthma',
 'hx_copd',
 'hx_heart_failure',
 'hx_atrial_fibrillation',
 'hx_ckd',
 'hx_liver_disease',
 'hx_malignancy',
 'hx_obesity',
 'hx_depression',
 'hx_anxiety',
 'hx_dementia',
 'hx_epilepsy',
 'hx_hypothyroidism',
 'hx_hyperthyroidism',
 'hx_hiv',
 'hx_coagulopathy',
 'hx_immunosuppressed',
 'hx_pregnant',
 'hx_substance_use_disorder',
 'hx_coronary_artery_disease',
 'hx_stroke_prior',
 'hx_peripheral_vascular_disease']

In [33]:
candidate_cols = [col for col in candidate_cols if col != "chief_complaint_raw"]

candidate_cols

['site_id',
 'arrival_mode',
 'arrival_hour',
 'arrival_day',
 'arrival_month',
 'arrival_season',
 'shift',
 'age_group',
 'sex',
 'language',
 'insurance_type',
 'transport_origin',
 'pain_location',
 'mental_status_triage',
 'chief_complaint_system',
 'num_prior_ed_visits_12m',
 'num_prior_admissions_12m',
 'num_active_medications',
 'num_comorbidities',
 'gcs_total',
 'pain_score',
 'hx_hypertension',
 'hx_diabetes_type2',
 'hx_diabetes_type1',
 'hx_asthma',
 'hx_copd',
 'hx_heart_failure',
 'hx_atrial_fibrillation',
 'hx_ckd',
 'hx_liver_disease',
 'hx_malignancy',
 'hx_obesity',
 'hx_depression',
 'hx_anxiety',
 'hx_dementia',
 'hx_epilepsy',
 'hx_hypothyroidism',
 'hx_hyperthyroidism',
 'hx_hiv',
 'hx_coagulopathy',
 'hx_immunosuppressed',
 'hx_pregnant',
 'hx_substance_use_disorder',
 'hx_coronary_artery_disease',
 'hx_stroke_prior',
 'hx_peripheral_vascular_disease']

In [34]:
def crear_tabla_enriquecimiento_pp(
    df,
    pp_col="pp_category_label",
    candidate_cols=None,
    min_pct_en_categoria=40,
    min_n_categoria=30
):
    resultados = []
    
    if candidate_cols is None:
        candidate_cols = []
    
    for pp_cat in sorted(df[pp_col].dropna().unique()):
        df_cat = df[df[pp_col] == pp_cat]
        n_pp_cat = len(df_cat)
        
        if n_pp_cat < min_n_categoria:
            continue
        
        for col in candidate_cols:
            # Saltar columnas completamente vacías
            if df[col].dropna().empty:
                continue
            
            # Para cada valor posible dentro de esa variable
            valores = df[col].dropna().unique()
            
            for valor in valores:
                # Proporción dentro de categoría PP
                n_valor_en_categoria = (df_cat[col] == valor).sum()
                pct_en_categoria = n_valor_en_categoria / n_pp_cat
                
                # Proporción global
                n_valor_global = (df[col] == valor).sum()
                pct_global = n_valor_global / len(df)
                
                # Evitar división por cero
                if pct_global == 0:
                    ratio_vs_global = np.nan
                else:
                    ratio_vs_global = pct_en_categoria / pct_global
                
                resultados.append({
                    "pp_category": pp_cat,
                    "n_pp_category": n_pp_cat,
                    "variable": col,
                    "valor": valor,
                    "n_valor_en_categoria": n_valor_en_categoria,
                    "pct_en_categoria": pct_en_categoria,
                    "pct_global": pct_global,
                    "diferencia_absoluta": pct_en_categoria - pct_global,
                    "ratio_vs_global": ratio_vs_global
                })
    
    tabla = pd.DataFrame(resultados)
    
    if tabla.empty:
        return tabla
    
    # Pasar a porcentaje
    tabla["pct_en_categoria_%"] = (tabla["pct_en_categoria"] * 100).round(2)
    tabla["pct_global_%"] = (tabla["pct_global"] * 100).round(2)
    tabla["diferencia_absoluta_%"] = (tabla["diferencia_absoluta"] * 100).round(2)
    tabla["ratio_vs_global"] = tabla["ratio_vs_global"].round(2)
    
    # Filtrar por sobrerrepresentación fuerte
    tabla_filtrada = tabla[
        (tabla["pct_en_categoria_%"] >= min_pct_en_categoria) &
        (tabla["n_valor_en_categoria"] >= 10)
    ].copy()
    
    tabla_filtrada = tabla_filtrada.sort_values(
        by=["pp_category", "pct_en_categoria_%", "diferencia_absoluta_%", "ratio_vs_global"],
        ascending=[True, False, False, False]
    )
    
    columnas_finales = [
        "pp_category",
        "n_pp_category",
        "variable",
        "valor",
        "n_valor_en_categoria",
        "pct_en_categoria_%",
        "pct_global_%",
        "diferencia_absoluta_%",
        "ratio_vs_global"
    ]
    
    return tabla_filtrada[columnas_finales]

In [35]:
tabla_sobrerrepresentacion_pp = crear_tabla_enriquecimiento_pp(
    df=df,
    pp_col="pp_category_label",
    candidate_cols=candidate_cols,
    min_pct_en_categoria=40,
    min_n_categoria=30
)

tabla_sobrerrepresentacion_pp

,pp_category,n_pp_category,variable,valor,n_valor_en_categoria,pct_en_categoria_%,pct_global_%,diferencia_absoluta_%,ratio_vs_global
182,PA faltante,4146,gcs_total,15,4146,100.00,81.60,18.40,1.23
246,PA faltante,4146,hx_pregnant,0,4020,96.96,96.46,0.50,1.01
129,PA faltante,4146,num_prior_admissions_12m,0,3549,85.60,71.33,14.27,1.20
99,PA faltante,4146,mental_status_triage,alert,3453,83.29,57.76,25.52,1.44
237,PA faltante,4146,hx_hypothyroidism,0,3420,82.49,80.32,2.17,1.03
...,...,...,...,...,...,...,...,...,...
1861,PP ≤ 0 imposible,2105,language,Finnish,1168,55.49,55.17,0.32,1.01
1974,PP ≤ 0 imposible,2105,gcs_total,15,1132,53.78,81.60,-27.82,0.66
1859,PP ≤ 0 imposible,2105,sex,F,1087,51.64,50.42,1.22,1.02
1797,PP ≤ 0 imposible,2105,arrival_mode,walk-in,1042,49.50,48.07,1.43,1.03


In [36]:
top_sobrerrepresentacion_por_pp = (
    tabla_sobrerrepresentacion_pp
    .groupby("pp_category", group_keys=False)
    .head(10)
)

top_sobrerrepresentacion_por_pp

,pp_category,n_pp_category,variable,valor,n_valor_en_categoria,pct_en_categoria_%,pct_global_%,diferencia_absoluta_%,ratio_vs_global
182,PA faltante,4146,gcs_total,15,4146,100.00,81.60,18.40,1.23
246,PA faltante,4146,hx_pregnant,0,4020,96.96,96.46,0.50,1.01
129,PA faltante,4146,num_prior_admissions_12m,0,3549,85.60,71.33,14.27,1.20
99,PA faltante,4146,mental_status_triage,alert,3453,83.29,57.76,25.52,1.44
237,PA faltante,4146,hx_hypothyroidism,0,3420,82.49,80.32,2.17,1.03
...,...,...,...,...,...,...,...,...,...
2004,PP ≤ 0 imposible,2105,hx_asthma,0,1692,80.38,80.35,0.03,1.00
2019,PP ≤ 0 imposible,2105,hx_obesity,0,1691,80.33,80.33,0.01,1.00
2029,PP ≤ 0 imposible,2105,hx_hypothyroidism,0,1691,80.33,80.32,0.01,1.00
2006,PP ≤ 0 imposible,2105,hx_copd,0,1690,80.29,80.34,-0.05,1.00


In [37]:
df["pp_category_label"].unique()

array(['PP 20-30 baja plausible', 'PP 30-60 usual', 'PP 10-20 muy baja',
       'PP 60-100 amplia', 'PA faltante', 'PP 0-10 extremadamente baja',
       'PP >100 extremadamente amplia', 'PP ≤ 0 imposible'], dtype=object)

In [38]:
categorias_pp_problematicas = [
    "PP ≤ 0 imposible",
    "PP 0-10 extremadamente baja",
    "PP 10-20 muy baja",
    "PP 20-30 baja plausible",
    "PP >100 extremadamente amplia"
]

tabla_sobrerrepresentacion_problematicas = tabla_sobrerrepresentacion_pp[
    tabla_sobrerrepresentacion_pp["pp_category"].isin(categorias_pp_problematicas)
]

tabla_sobrerrepresentacion_problematicas

,pp_category,n_pp_category,variable,valor,n_valor_en_categoria,pct_en_categoria_%,pct_global_%,diferencia_absoluta_%,ratio_vs_global
502,PP 0-10 extremadamente baja,2703,hx_pregnant,0,2591,95.86,96.46,-0.61,0.99
488,PP 0-10 extremadamente baja,2703,hx_dementia,0,2277,84.24,80.90,3.34,1.04
470,PP 0-10 extremadamente baja,2703,hx_copd,0,2224,82.28,80.34,1.94,1.02
466,PP 0-10 extremadamente baja,2703,hx_diabetes_type1,0,2188,80.95,80.85,0.10,1.00
510,PP 0-10 extremadamente baja,2703,hx_peripheral_vascular_disease,0,2185,80.84,80.36,0.48,1.01
...,...,...,...,...,...,...,...,...,...
1861,PP ≤ 0 imposible,2105,language,Finnish,1168,55.49,55.17,0.32,1.01
1974,PP ≤ 0 imposible,2105,gcs_total,15,1132,53.78,81.60,-27.82,0.66
1859,PP ≤ 0 imposible,2105,sex,F,1087,51.64,50.42,1.22,1.02
1797,PP ≤ 0 imposible,2105,arrival_mode,walk-in,1042,49.50,48.07,1.43,1.03


In [39]:
hx_cols = [col for col in df.columns if col.startswith("hx_")]

prevalencia_global_hx = df[hx_cols].mean()

enriquecimiento_hx_por_pp = []

for categoria in sorted(df["pp_category_label"].dropna().unique()):
    temp = df[df["pp_category_label"] == categoria]
    
    prevalencia_categoria = temp[hx_cols].mean()
    
    resumen = pd.DataFrame({
        "antecedente": hx_cols,
        "prevalencia_global": prevalencia_global_hx.values,
        "prevalencia_en_categoria": prevalencia_categoria.values
    })
    
    resumen["diferencia_absoluta"] = resumen["prevalencia_en_categoria"] - resumen["prevalencia_global"]
    resumen["ratio_vs_global"] = resumen["prevalencia_en_categoria"] / resumen["prevalencia_global"]
    
    resumen["pp_category_label"] = categoria
    resumen["n_categoria"] = len(temp)
    
    resumen["prevalencia_global_%"] = (resumen["prevalencia_global"] * 100).round(2)
    resumen["prevalencia_en_categoria_%"] = (resumen["prevalencia_en_categoria"] * 100).round(2)
    resumen["diferencia_absoluta_%"] = (resumen["diferencia_absoluta"] * 100).round(2)
    resumen["ratio_vs_global"] = resumen["ratio_vs_global"].round(2)
    
    enriquecimiento_hx_por_pp.append(resumen)

enriquecimiento_hx_por_pp = pd.concat(enriquecimiento_hx_por_pp, ignore_index=True)

enriquecimiento_hx_por_pp = enriquecimiento_hx_por_pp[
    [
        "pp_category_label",
        "n_categoria",
        "antecedente",
        "prevalencia_en_categoria_%",
        "prevalencia_global_%",
        "diferencia_absoluta_%",
        "ratio_vs_global"
    ]
]

# Ordenar por categoría y mayor prevalencia dentro de esa categoría
enriquecimiento_hx_por_pp_ordenado = enriquecimiento_hx_por_pp.sort_values(
    by=["pp_category_label", "prevalencia_en_categoria_%", "diferencia_absoluta_%"],
    ascending=[True, False, False]
)

enriquecimiento_hx_por_pp_ordenado

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global
0,PA faltante,4146,hx_hypertension,27.18,28.16,-0.98,0.97
5,PA faltante,4146,hx_heart_failure,27.06,28.10,-1.03,0.96
7,PA faltante,4146,hx_ckd,27.04,28.18,-1.14,0.96
22,PA faltante,4146,hx_coronary_artery_disease,26.92,28.26,-1.35,0.95
6,PA faltante,4146,hx_atrial_fibrillation,26.75,28.21,-1.46,0.95
...,...,...,...,...,...,...,...
183,PP ≤ 0 imposible,2105,hx_liver_disease,19.57,19.92,-0.35,0.98
199,PP ≤ 0 imposible,2105,hx_peripheral_vascular_disease,19.48,19.64,-0.17,0.99
177,PP ≤ 0 imposible,2105,hx_diabetes_type1,18.72,19.15,-0.43,0.98
188,PP ≤ 0 imposible,2105,hx_dementia,14.16,19.10,-4.94,0.74


In [40]:
top_hx_por_pp_mayor_porcentaje = (
    enriquecimiento_hx_por_pp_ordenado
    .groupby("pp_category_label", group_keys=False)
    .head(10)
)

top_hx_por_pp_mayor_porcentaje

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global
0,PA faltante,4146,hx_hypertension,27.18,28.16,-0.98,0.97
5,PA faltante,4146,hx_heart_failure,27.06,28.10,-1.03,0.96
7,PA faltante,4146,hx_ckd,27.04,28.18,-1.14,0.96
22,PA faltante,4146,hx_coronary_artery_disease,26.92,28.26,-1.35,0.95
6,PA faltante,4146,hx_atrial_fibrillation,26.75,28.21,-1.46,0.95
...,...,...,...,...,...,...,...
198,PP ≤ 0 imposible,2105,hx_stroke_prior,24.32,27.76,-3.44,0.88
182,PP ≤ 0 imposible,2105,hx_ckd,23.56,28.18,-4.62,0.84
194,PP ≤ 0 imposible,2105,hx_immunosuppressed,21.14,20.08,1.06,1.05
191,PP ≤ 0 imposible,2105,hx_hyperthyroidism,20.67,19.93,0.74,1.04


In [41]:
hx_sobre_40_por_pp = enriquecimiento_hx_por_pp_ordenado[
    enriquecimiento_hx_por_pp_ordenado["prevalencia_en_categoria_%"] >= 40
]

hx_sobre_40_por_pp

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global


In [42]:
hx_sobre_40_enriquecidos = hx_sobre_40_por_pp[
    hx_sobre_40_por_pp["ratio_vs_global"] >= 1.2
]

hx_sobre_40_enriquecidos

,pp_category_label,n_categoria,antecedente,prevalencia_en_categoria_%,prevalencia_global_%,diferencia_absoluta_%,ratio_vs_global


In [43]:
from pathlib import Path

# Carpeta destino
output_dir = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed")

# Crear carpeta si no existe
output_dir.mkdir(parents=True, exist_ok=True)

# Nombre del nuevo checkpoint
output_path = output_dir / "train_full_bp_audit_v1.csv"

# Guardar dataset con las columnas nuevas
df.to_csv(output_path, index=False)

print(f"Archivo guardado en: {output_path}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Archivo guardado en: C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_bp_audit_v1.csv
Filas: 80000
Columnas: 76


In [44]:
df_check = pd.read_csv(output_path)

print(df_check.shape)
df_check.head()

(80000, 76)


,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,pp_recalc,bp_missing_original,pp_category,pp_impossible_flag,pp_extremely_low_flag,pp_very_low_flag,pp_low_flag,pp_wide_flag,pp_extremely_wide_flag,pp_category_label
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,21.5,False,04_pp_20_30_low_plausible,0,0,0,1,0,0,PP 20-30 baja plausible
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,38.3,False,05_pp_30_60_usual,0,0,0,0,0,0,PP 30-60 usual
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,11.4,False,03_pp_10_20_very_low,0,0,1,1,0,0,PP 10-20 muy baja
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,82.4,False,06_pp_60_100_wide,0,0,0,0,1,0,PP 60-100 amplia
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,64.7,False,06_pp_60_100_wide,0,0,0,0,1,0,PP 60-100 amplia


# Conclusiones auditoría de presión arterial y presión de pulso

## Objetivo

Evaluar la coherencia fisiológica y el comportamiento de la presión arterial y la presión de pulso antes de definir estrategias de imputación o modelado.

## Hallazgos principales

### 1. La ausencia de presión arterial parece ser un fenómeno distinto a la presión de pulso anómala

Los casos con presión arterial faltante se concentran casi exclusivamente en pacientes ESI 4 y ESI 5, presentan predominio de alta médica y no muestran fallecimientos. Esto sugiere que la ausencia de registro de PA podría reflejar baja acuidad, decisiones operacionales del triage o reglas de generación del dataset, más que gravedad clínica.

### 2. Las categorías extremas de presión de pulso muestran asociación con mayor gravedad

Las categorías de presión de pulso imposible (PP ≤ 0), extremadamente baja (0-10 mmHg) y muy baja (10-20 mmHg) presentan mayor proporción de pacientes ESI 1-2, hospitalizaciones, traslados y fallecimientos en comparación con las categorías habituales de presión de pulso.

### 3. La presión de pulso no parece distribuirse aleatoriamente

Las diferencias observadas entre categorías sugieren que la presión de pulso contiene información asociada a severidad clínica o a las reglas utilizadas para construir el dataset. Su comportamiento parece consistente con una señal fisiológica relevante y no con ruido completamente aleatorio.

### 4. No se identificó una comorbilidad única que explique las categorías extremas de presión de pulso

La revisión de antecedentes clínicos mostró que variables como fibrilación auricular, insuficiencia cardíaca, enfermedad renal crónica u otras comorbilidades no parecen explicar por sí solas las categorías más extremas de presión de pulso.

### 5. Las categorías de presión de pulso serán conservadas

Dado que las categorías extremas muestran asociaciones consistentes con indicadores de gravedad, no se eliminarán ni corregirán automáticamente en esta etapa. Se conservarán como variables de auditoría y potenciales predictores para etapas posteriores.

## Decisiones metodológicas derivadas

* No realizar intercambio automático de PAS y PAD.
* Mantener una categoría específica para PA faltante.
* Mantener flags de calidad para PP imposible y PP extrema.
* Posponer cualquier imputación hasta completar el análisis de missingness.
* Recalcular variables derivadas después de definir la estrategia de imputación de PAS y PAD.
